In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
 

In [59]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [60]:
CORPUS = """
To be or not to be that is the question whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune or to take arms against a sea of troubles
and by opposing end them to die to sleep no more and by a sleep to say we end
the heartache and the thousand natural shocks that flesh is heir to tis a consummation
devoutly to be wished to die to sleep to sleep perchance to dream ay there is the rub
for in that sleep of death what dreams may come when we have shuffled off this mortal coil
must give us pause there is the respect that makes calamity of so long life
""".strip().lower()

In [61]:
chars   = sorted(set(CORPUS))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
 
print(f"Vocabulary size : {vocab_size}")
print(f"Corpus length   : {len(CORPUS)} characters\n")
 

Vocabulary size : 25
Corpus length   : 584 characters



In [ ]:
class CharDataset(Dataset):
    
    def __init__(self, text: str, seq_len: int):
        self.seq_len = seq_len
        self.data    = [char2idx[c] for c in text]
 
    def __len__(self):
        return len(self.data) - self.seq_len
 
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx : idx + self.seq_len],          dtype=torch.long)
        y = torch.tensor(self.data[idx + 1 : idx + self.seq_len + 1],  dtype=torch.long)
        return x, y
 
 
SEQ_LEN    = 40
BATCH_SIZE = 32
 
dataset    = CharDataset(CORPUS, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
 
print(f"Number of sequences : {len(dataset)}")
print(f"Batches per epoch   : {len(dataloader)}\n")

Number of sequences : 544
Batches per epoch   : 17



In [ ]:
class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float = 0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
 
        self.embed   = nn.Embedding(vocab_size, embed_dim)
        self.lstm    = nn.LSTM(embed_dim, hidden_dim,
                               num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, vocab_size)
 
    def forward(self, x, hidden=None):

        emb    = self.dropout(self.embed(x))          
        out, hidden = self.lstm(emb, hidden)          
        out    = self.dropout(out)
        logits = self.fc(out)                          
        return logits, hidden
 
    def init_hidden(self, batch_size: int, device: torch.device):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        return (h, c)
 

In [64]:
EMBED_DIM  = 64
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT    = 0.3
LR         = 1e-3
EPOCHS     = 30
CLIP_GRAD  = 5.0
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")
 
model     = LSTMLanguageModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
 
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}\n")
print(model)
print()

Using device: cpu

Trainable parameters: 864,089

LSTMLanguageModel(
  (embed): Embedding(25, 64)
  (lstm): LSTM(64, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=25, bias=True)
)



In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, clip):
    model.train()
    total_loss = 0.0
 
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
 
        
        hidden = model.init_hidden(x.size(0), device)
 
        optimizer.zero_grad()
        logits, hidden = model(x, hidden)
 
        
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
 
        nn.utils.clip_grad_norm_(model.parameters(), clip)  
        optimizer.step()
 
        total_loss += loss.item()
 
    return total_loss / len(dataloader)
 
 
print("=" * 55)
print(f"{'Epoch':>6}  {'Loss':>8}  {'Perplexity':>12}  {'LR':>10}")
print("=" * 55)
 
history = {"loss": [], "perplexity": []}
 
for epoch in range(1, EPOCHS + 1):
    avg_loss   = train_epoch(model, dataloader, criterion, optimizer, device, CLIP_GRAD)
    perplexity = np.exp(avg_loss)
    current_lr = scheduler.get_last_lr()[0]
 
    history["loss"].append(avg_loss)
    history["perplexity"].append(perplexity)
 
    print(f"{epoch:>6}  {avg_loss:>8.4f}  {perplexity:>12.2f}  {current_lr:>10.6f}")
    scheduler.step()
 
print("=" * 55)

 Epoch      Loss    Perplexity          LR
     1    2.9339         18.80    0.001000
     2    2.6608         14.31    0.001000
     3    2.2666          9.65    0.001000
     4    1.8749          6.52    0.001000
     5    1.5229          4.59    0.001000
     6    1.1825          3.26    0.001000
     7    0.8798          2.41    0.001000
     8    0.6427          1.90    0.001000
     9    0.4817          1.62    0.001000
    10    0.3700          1.45    0.001000
    11    0.3199          1.38    0.000500
    12    0.2917          1.34    0.000500
    13    0.2697          1.31    0.000500
    14    0.2585          1.29    0.000500
    15    0.2419          1.27    0.000500
    16    0.2316          1.26    0.000500
    17    0.2214          1.25    0.000500
    18    0.2124          1.24    0.000500
    19    0.2031          1.23    0.000500
    20    0.1993          1.22    0.000500
    21    0.1935          1.21    0.000250
    22    0.1893          1.21    0.000250
    23    0

In [66]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
 
    with torch.no_grad():
        for x, y in dataloader:
            x, y   = x.to(device), y.to(device)
            hidden = model.init_hidden(x.size(0), device)
            logits, _ = model(x, hidden)
            loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            total_loss += loss.item()
 
    avg_loss   = total_loss / len(dataloader)
    perplexity = np.exp(avg_loss)
    return avg_loss, perplexity
 
 
eval_loss, eval_ppl = evaluate(model, dataloader, criterion, device)
print(f"\n[Evaluation] Loss: {eval_loss:.4f}  |  Perplexity: {eval_ppl:.2f}\n")


[Evaluation] Loss: 0.1252  |  Perplexity: 1.13



In [ ]:
def generate(model, seed_text: str, num_chars: int = 200,
             temperature: float = 1.0, device=device) -> str:
    model.eval()
    generated = seed_text
 
    
    input_ids = torch.tensor(
        [[char2idx.get(c, 0) for c in seed_text]], dtype=torch.long
    ).to(device)
 
    hidden = model.init_hidden(1, device)
 
    with torch.no_grad():

        _, hidden = model(input_ids, hidden)
 
        x = input_ids[:, -1:]          
 
        for _ in range(num_chars):
            logits, hidden = model(x, hidden)          
            logits = logits[:, -1, :] / temperature    
            probs  = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            generated += idx2char[next_id.item()]
            x = next_id
 
    return generated
 
 
seed = "to be or not to be"
print("─" * 55)
print("GENERATION  (temperature=0.5 — focused)")
print("─" * 55)
print(generate(model, seed, num_chars=250, temperature=0.5))
 
print("\n" + "─" * 55)
print("GENERATION  (temperature=1.0 — balanced)")
print("─" * 55)
print(generate(model, seed, num_chars=250, temperature=1.0))
 
print("\n" + "─" * 55)
print("GENERATION  (temperature=1.5 — creative)")
print("─" * 55)
print(generate(model, seed, num_chars=250, temperature=1.5))

───────────────────────────────────────────────────────
GENERATION  (temperature=0.5 — focused)
───────────────────────────────────────────────────────
to be or not to be that is the question whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune or to take arms against a sea of troubles
and by opposing end them to die to sleep no more and by a sleep to say we end
the heartache and the t

───────────────────────────────────────────────────────
GENERATION  (temperature=1.0 — balanced)
───────────────────────────────────────────────────────
to be or not to be that is the questie to sleep to sleep perchance to dream ay there is the rub
for in that sleep of death what dreams may come when we have shuffled off thus mortal coil
must give us pause there is the respect that makes calamity of so long ling ars a

───────────────────────────────────────────────────────
GENERATION  (temperature=1.5 — creative)
───────────────────────────────────────────────────────
to